# Crop Disease Detection
**MobileNetV2 feature extraction + XGBoost classifier on PlantVillage**

Pipeline:
1. Load PlantVillage dataset (Pepper × 2, Potato × 3, Tomato × 10 = 15 classes)
2. Extract 1280-d features with frozen MobileNetV2 + GlobalAveragePooling2D
3. Train XGBoost baseline
4. Tune hyperparameters with Optuna
5. Evaluate: accuracy, macro F1, top-3 accuracy, confusion matrix

> **Dataset:** Add `emmarex/plantdisease` via *Add Data* before running.
> **CPU runtime:** ~25–35 min total (feature extraction ~20 min, Optuna tuning ~10 min).

## 1 — Install dependencies

In [ ]:
%%capture
!pip install xgboost optuna

## 2 — Imports & config

In [ ]:
import math
import os
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import tensorflow as tf
import xgboost as xgb
from PIL import Image
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"TF {tf.__version__}  |  XGB {xgb.__version__}  |  Optuna {optuna.__version__}")

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
DATA_DIR      = Path("/kaggle/input/plantdisease/PlantVillage")
OUTPUT_DIR    = Path("/kaggle/working")

INPUT_SIZE    = (128, 128)
NUM_CHANNELS  = 3
FEATURE_DIM   = 1280          # MobileNetV2 GAP output
NUM_CLASSES   = 15
BATCH_SIZE    = 32            # Reduced from 64 — safe for CPU memory

TRAIN_SPLIT   = 0.70
VAL_SPLIT     = 0.15
RANDOM_STATE  = 42

# CPU-safe Optuna settings (GPU: raise N_TRIALS→50, CV_FOLDS→3)
N_TRIALS      = 15
CV_FOLDS      = 2

IMG_EXTENSIONS = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}

# XGBoost baseline
XGB_BASE = dict(
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cpu",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    reg_lambda=1.0,
    reg_alpha=0.1,
)

OUTPUT_DIR.mkdir(exist_ok=True)
print("Config OK")

## 3 — Dataset scan & stratified split

In [ ]:
def scan_dataset(data_dir: Path) -> pd.DataFrame:
    class_names = sorted(d.name for d in data_dir.iterdir() if d.is_dir())
    class_index = {name: idx for idx, name in enumerate(class_names)}
    rows = []
    for name, idx in class_index.items():
        for p in (data_dir / name).iterdir():
            if p.suffix in IMG_EXTENSIONS:
                rows.append({"filepath": str(p), "label_str": name, "label_int": idx})
    df = pd.DataFrame(rows)
    print(f"Scanned {len(df):,} images across {df['label_int'].nunique()} classes")
    return df, class_index


def stratified_split(df):
    test_frac = 1.0 - TRAIN_SPLIT
    train_df, temp_df = train_test_split(
        df, test_size=test_frac, stratify=df["label_int"], random_state=RANDOM_STATE
    )
    val_frac = VAL_SPLIT / test_frac
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_frac), stratify=temp_df["label_int"], random_state=RANDOM_STATE
    )
    print(f"Split — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


df, class_index = scan_dataset(DATA_DIR)
train_df, val_df, test_df = stratified_split(df)

# Label encoder for inverse_transform during inference
le = LabelEncoder()
le.fit(sorted(class_index.keys()))
label_names = list(le.classes_)

# Class distribution
df.groupby("label_str").size().sort_values().plot(
    kind="barh", figsize=(9, 6), title="Images per class", color="steelblue"
)
plt.xlabel("Count")
plt.tight_layout()
plt.show()

## 4 — MobileNetV2 feature extractor

In [ ]:
def build_feature_model() -> tf.keras.Model:
    """Frozen MobileNetV2 + GlobalAveragePooling2D → (B, 1280) features."""
    base = tf.keras.applications.MobileNetV2(
        input_shape=(*INPUT_SIZE, NUM_CHANNELS),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    gap = tf.keras.layers.GlobalAveragePooling2D(name="gap")(base.output)
    model = tf.keras.Model(inputs=base.input, outputs=gap)
    total = sum(tf.size(v).numpy() for v in model.variables)
    print(f"Feature model — {total:,} total params, 0 trainable (fully frozen)")
    return model


def build_augmentation_pipeline():
    return tf.keras.Sequential([
        tf.keras.layers.Resizing(*INPUT_SIZE),
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomBrightness(0.20),
        tf.keras.layers.RandomContrast(0.20),
        tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
        tf.keras.layers.GaussianNoise(0.05),
    ], name="aug_train")


def build_preprocessing_pipeline():
    return tf.keras.Sequential([
        tf.keras.layers.Resizing(*INPUT_SIZE),
        tf.keras.layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    ], name="preprocess_eval")


feature_model = build_feature_model()
aug_pipe  = build_augmentation_pipeline()
eval_pipe = build_preprocessing_pipeline()

## 5 — Feature extraction (cached to disk)

In [ ]:
def load_images_batch(file_list, batch_size=BATCH_SIZE):
    h, w = INPUT_SIZE
    for start in range(0, len(file_list), batch_size):
        batch_files = file_list[start:start + batch_size]
        imgs = []
        for fp in batch_files:
            try:
                img = Image.open(fp).convert("RGB").resize((w, h), Image.LANCZOS)
                imgs.append(np.array(img, dtype=np.float32))
            except Exception:
                imgs.append(np.zeros((h, w, 3), dtype=np.float32))
        yield np.stack(imgs, axis=0)


def extract_features(split_df, pipe, desc="Extracting"):
    files  = split_df["filepath"].tolist()
    labels = split_df["label_int"].values
    n_batches = math.ceil(len(files) / BATCH_SIZE)
    all_feats, all_labels = [], []
    for i, batch in enumerate(tqdm(load_images_batch(files), total=n_batches, desc=desc)):
        batch_pp = pipe(batch, training=True).numpy()
        feats = feature_model.predict_on_batch(batch_pp)
        start = i * BATCH_SIZE
        all_feats.append(feats)
        all_labels.append(labels[start:start + len(batch)])
    X = np.vstack(all_feats).astype(np.float32)
    y = np.concatenate(all_labels).astype(np.int32)
    print(f"  {desc}: X={X.shape}  y={y.shape}")
    return X, y


TRAIN_CACHE = OUTPUT_DIR / "train_features_128.npz"
VAL_CACHE   = OUTPUT_DIR / "val_features_128.npz"
TEST_CACHE  = OUTPUT_DIR / "test_features_128.npz"


def load_or_extract(split_df, pipe, cache_path, desc):
    if cache_path.exists():
        data = np.load(cache_path)
        print(f"Loaded from cache: {cache_path.name}")
        return data["X"], data["y"]
    X, y = extract_features(split_df, pipe, desc=desc)
    np.savez_compressed(cache_path, X=X, y=y)
    print(f"Saved to {cache_path.name} ({cache_path.stat().st_size/1e6:.1f} MB)")
    return X, y


X_train, y_train = load_or_extract(train_df, aug_pipe,  TRAIN_CACHE, "Train")
X_val,   y_val   = load_or_extract(val_df,   eval_pipe, VAL_CACHE,   "Val")
X_test,  y_test  = load_or_extract(test_df,  eval_pipe, TEST_CACHE,  "Test")

## 6 — XGBoost baseline

In [ ]:
print("Training XGBoost baseline...")
baseline = xgb.XGBClassifier(**XGB_BASE)
baseline.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

val_f1 = f1_score(y_val, baseline.predict(X_val), average="macro")
print(f"\nBaseline val macro F1: {val_f1:.4f}")

## 7 — Optuna hyperparameter tuning

In [ ]:
def make_objective(X, y):
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = dict(
            objective="multi:softprob",
            num_class=NUM_CLASSES,
            eval_metric="mlogloss",
            tree_method="hist",
            device="cpu",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            n_estimators=trial.suggest_int("n_estimators", 100, 400),
            learning_rate=trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 6),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            reg_lambda=trial.suggest_float("reg_lambda", 0.5, 5.0),
            reg_alpha=trial.suggest_float("reg_alpha", 0.0, 2.0),
        )
        scores = cross_val_score(
            xgb.XGBClassifier(**params), X, y, cv=skf, scoring="f1_macro", n_jobs=1
        )
        return scores.mean()

    return objective


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=max(3, N_TRIALS // 3)),
)
study.optimize(make_objective(X_train, y_train), n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"\nBest trial #{best.number} — macro F1: {best.value:.4f}")
print("Best params:", best.params)

## 8 — Retrain with best hyperparameters

In [ ]:
tuned_params = dict(
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cpu",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    **study.best_params,
)

final_model = xgb.XGBClassifier(**tuned_params)
final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

# Save model
model_path = OUTPUT_DIR / "xgb_crop_disease.json"
final_model.save_model(str(model_path))
print(f"Model saved to {model_path}")

## 9 — Test set evaluation

In [ ]:
def top_k_accuracy(y_true, y_proba, k=3):
    top_k = np.argsort(y_proba, axis=1)[:, -k:]
    return sum(y_true[i] in top_k[i] for i in range(len(y_true))) / len(y_true)


y_pred  = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

metrics = dict(
    accuracy   = accuracy_score(y_test, y_pred),
    macro_f1   = f1_score(y_test, y_pred, average="macro"),
    weighted_f1= f1_score(y_test, y_pred, average="weighted"),
    top3_acc   = top_k_accuracy(y_test, y_proba, k=3),
)

print("=" * 42)
print("TEST SET RESULTS")
print("=" * 42)
for k, v in metrics.items():
    print(f"  {k:<15} {v:.4f}")
print("=" * 42)

## 10 — Classification report

In [ ]:
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

## 11 — Confusion matrix

In [ ]:
def shorten(name):
    return re.sub(r"_+", " ", name).replace("Two spotted spider mite", "Spider mites")


short_names = [shorten(n) for n in label_names]
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=short_names, yticklabels=short_names, ax=ax,
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion Matrix — Crop Disease Classifier (15 classes)", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 12 — Per-class F1 chart

In [ ]:
report_dict = __import__("sklearn.metrics", fromlist=["classification_report"]).classification_report(
    y_test, y_pred, target_names=label_names, output_dict=True, zero_division=0
)

CROP_COLORS = {"Tomato": "#d62728", "Potato": "#8c564b", "Pepper": "#ff7f0e"}

f1_items = sorted(
    [(n, report_dict[n]["f1-score"]) for n in label_names if n in report_dict],
    key=lambda x: x[1],
)
names  = [shorten(k) for k, _ in f1_items]
values = [v for _, v in f1_items]
colors = [CROP_COLORS.get(re.split(r"_+", k)[0], "#1f77b4") for k, _ in f1_items]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(names, values, color=colors)
ax.axvline(0.9, color="red", linestyle="--", linewidth=0.8, label="F1 = 0.90")
ax.set_xlim(0, 1.0)
ax.set_xlabel("F1 Score (macro)")
ax.set_title("Per-Class F1 Score (worst → best)")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

## 13 — Optuna optimisation history

In [ ]:
trial_numbers = [t.number for t in study.trials]
trial_values  = [t.value  for t in study.trials]
best_so_far   = pd.Series(trial_values).cummax().tolist()

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(trial_numbers, trial_values, alpha=0.5, label="Trial F1", s=20)
ax.plot(trial_numbers, best_so_far, color="red", linewidth=1.5, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("Macro F1")
ax.set_title("Optuna Optimisation History")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "optuna_history.png", dpi=120, bbox_inches="tight")
plt.show()

## 14 — Inference helper

In [ ]:
def predict_disease(image_path: str, top_k: int = 3) -> list[dict]:
    """Run a single field photo through the pipeline and return top-k predictions."""
    h, w = INPUT_SIZE
    img = Image.open(image_path).convert("RGB").resize((w, h), Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    arr = tf.keras.applications.mobilenet_v2.preprocess_input(arr)
    arr = arr[np.newaxis, ...]                                  # (1, H, W, 3)

    features = feature_model.predict(arr, verbose=0)            # (1, 1280)
    probas   = final_model.predict_proba(features)[0]           # (15,)
    top_idx  = np.argsort(probas)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, 1):
        raw = le.inverse_transform([idx])[0]
        parts = re.split(r"_+", raw)
        crop    = parts[0]
        disease = " ".join(parts[1:]) if len(parts) > 1 else "Unknown"
        results.append(dict(rank=rank, crop=crop, disease=disease, confidence=float(probas[idx])))
    return results


# Example — swap path with any test image
sample_path = test_df.iloc[0]["filepath"]
preds = predict_disease(sample_path)

print("Crop Disease Classification Result")
print("=" * 37)
for p in preds:
    print(f"  #{p['rank']}  {p['crop']} — {p['disease']:<28}  {p['confidence']*100:5.1f}%")

## 15 — Output files

In [ ]:
print("Files saved to /kaggle/working:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size/1e3:8.1f} KB")